# Análise do `time_scale` aprendido pelo HoTHP

**Objetivo:** verificar se o parâmetro `log_time_scale` (escala temporal aprendida via backprop)
do kernel hiperbólico se adapta corretamente a diferentes regimes temporais.

**Protocolo:** mesmo de `validacao_pos_correcao.ipynb` — *train short, test long* com dados sintéticos de Hawkes.
- Decaimento **lento** (β ≈ 0.025) → inter-event times maiores
- Decaimento **rápido** (β ≈ 0.50) → inter-event times menores
- Fatores de extrapolação: α ∈ {1, 2, 5, 10}

**Parâmetros monitorados:**
- `time_scale = exp(log_time_scale)` — escala aprendida para `Δt` antes do kernel hiperbólico
- `theta_prime` — coeficiente de decaimento global (condição: θ' > max(θⱼ))
- `theta_prime_raw` — valor bruto antes da transformação

**Hipótese:** o `time_scale` deve convergir para valores que normalizam a escala temporal
do dataset, tornando o kernel hiperbólico efetivo independente da escala absoluta dos tempos.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, sys, math, random, json, pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('ufc-easytpp'):
        os.system('git clone https://github.com/hugoramos/ufc-easytpp.git')

    # ── Sobrescrever torch_hothp.py com versao atual (time_scale, sem clamp) ─
    HOTHP_CODE = r'''import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from easy_tpp.model.torch_model.torch_baselayer import MultiHeadAttention, EncoderLayer, attention, ScaledSoftplus
from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP


class HyperbolicRotaryEmbedding(nn.Module):
    def __init__(self, dim, max_freq=10000):
        super().__init__()
        self.dim = dim
        self.max_freq = max_freq
        thetas = torch.tensor([
            max_freq ** (-2.0 * (j - 1) / dim) for j in range(1, dim // 2 + 1)
        ])
        self.register_buffer('thetas', thetas)
        self.theta_prime_raw = nn.Parameter(torch.tensor(0.5))
        self.log_time_scale = nn.Parameter(torch.tensor(0.0))

    @property
    def theta_prime(self):
        return F.softplus(self.theta_prime_raw) + self.thetas.max().item() + 1e-4

    @property
    def time_scale(self):
        return torch.exp(self.log_time_scale)


def hyperbolic_attention(q, k, v, time_seqs, thetas, theta_prime, time_scale, mask=None, dropout=None, chunk_size=16):
    d_k = q.shape[-1]
    B, H, L, _ = q.shape
    q1 = q[..., 0::2]
    q2 = q[..., 1::2]
    k1 = k[..., 0::2]
    k2 = k[..., 1::2]
    k1_j = k1.unsqueeze(2)
    k2_j = k2.unsqueeze(2)
    t_j = time_seqs.unsqueeze(1)
    thetas_v = thetas.view(1, 1, -1)
    att_scores = torch.zeros(B, H, L, L, device=q.device, dtype=q.dtype)
    for i_start in range(0, L, chunk_size):
        i_end = min(i_start + chunk_size, L)
        t_i = time_seqs[:, i_start:i_end].unsqueeze(-1)
        delta = (t_i - t_j) * time_scale
        abs_d  = delta.abs().unsqueeze(-1)
        sign_d = delta.sign().unsqueeze(-1)
        exp_arg_m = -abs_d * (theta_prime - thetas_v)
        exp_arg_p = -abs_d * (theta_prime + thetas_v)
        exp_m = torch.exp(exp_arg_m)
        exp_p = torch.exp(exp_arg_p)
        dc = ((exp_m + exp_p) / 2).unsqueeze(1)
        ds = (sign_d * (exp_m - exp_p) / 2).unsqueeze(1)
        q1_i = q1[:, :, i_start:i_end, :].unsqueeze(3)
        q2_i = q2[:, :, i_start:i_end, :].unsqueeze(3)
        att_scores[:, :, i_start:i_end, :] = (
            (q1_i * k1_j + q2_i * k2_j) * dc +
            (q1_i * k2_j + q2_i * k1_j) * ds
        ).sum(-1)
    att_scores = att_scores / math.sqrt(d_k)
    if mask is not None:
        att_scores = att_scores.masked_fill(mask > 0, -1e4)
    att_weights = torch.softmax(att_scores, dim=-1)
    if dropout is not None:
        att_weights = dropout(att_weights)
    return torch.matmul(att_weights, v), att_weights


class HyperbolicMultiHeadAttention(MultiHeadAttention):
    def forward(self, query, key, value, mask, time_seqs=None, thetas=None, theta_prime=None, time_scale=None, output_weight=False):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin_layer(x).view(nbatches, -1, self.n_head, self.d_k).transpose(1, 2)
            for lin_layer, x in zip(self.linears, (query, key, value))
        ]
        x, attn_weight = hyperbolic_attention(query, key, value, time_seqs, thetas, theta_prime, time_scale, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.n_head * self.d_k)
        if self.output_linear:
            return (self.linears[-1](x), attn_weight) if output_weight else self.linears[-1](x)
        else:
            return (x, attn_weight) if output_weight else x


class HyperbolicEncoderLayer(EncoderLayer):
    def forward(self, x, mask, time_seqs=None, thetas=None, theta_prime=None, time_scale=None):
        if self.use_residual:
            x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask, time_seqs=time_seqs, thetas=thetas, theta_prime=theta_prime, time_scale=time_scale))
            return self.sublayer[1](x, self.feed_forward) if self.feed_forward is not None else x
        else:
            x = self.self_attn(x, x, x, mask, time_seqs=time_seqs, thetas=thetas, theta_prime=theta_prime, time_scale=time_scale)
            return self.feed_forward(x) if self.feed_forward is not None else x


class HoTHP(THP):
    def __init__(self, model_config):
        TorchBaseModel.__init__(self, model_config)
        self.d_model = model_config.hidden_size
        self.d_time = model_config.time_emb_size
        self.use_norm = model_config.use_ln
        self.n_layers = model_config.num_layers
        self.n_head = model_config.num_heads
        self.dropout = model_config.dropout_rate
        self.hope_emb = HyperbolicRotaryEmbedding(self.d_model // self.n_head)
        self.factor_intensity_base = nn.Parameter(torch.empty([1, self.num_event_types], device=self.device))
        self.factor_intensity_decay = nn.Parameter(torch.empty([1, self.num_event_types], device=self.device))
        nn.init.xavier_normal_(self.factor_intensity_base)
        nn.init.xavier_normal_(self.factor_intensity_decay)
        self.layer_intensity_hidden = nn.Linear(self.d_model, self.num_event_types)
        self.softplus = ScaledSoftplus(self.num_event_types)
        self.feed_forward = nn.Sequential(
            nn.Linear(self.d_model, self.d_model * 2),
            nn.ReLU(),
            nn.Linear(self.d_model * 2, self.d_model)
        )
        self.stack_layers = nn.ModuleList([
            HyperbolicEncoderLayer(
                self.d_model,
                HyperbolicMultiHeadAttention(self.n_head, self.d_model, self.d_model, self.dropout, output_linear=False),
                use_residual=False, feed_forward=self.feed_forward, dropout=self.dropout
            ) for _ in range(self.n_layers)
        ])

    def _normalize_timestamps(self, time_seqs):
        return time_seqs - time_seqs[:, :1]

    def forward(self, time_seqs, type_seqs, attention_mask):
        norm_times = self._normalize_timestamps(time_seqs)
        enc_output = self.layer_type_emb(type_seqs)
        thetas = self.hope_emb.thetas
        theta_prime = self.hope_emb.theta_prime
        time_scale = self.hope_emb.time_scale
        for enc_layer in self.stack_layers:
            enc_output = enc_layer(enc_output, mask=attention_mask, time_seqs=norm_times,
                                   thetas=thetas, theta_prime=theta_prime, time_scale=time_scale)
        return enc_output
'''

    hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
    with open(hothp_path, 'w') as f:
        f.write(HOTHP_CODE)
    print('torch_hothp.py sobrescrito com versao atual (time_scale, sem clamp)')

    # Fix de importacao do __init__
    init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
    with open(init_path, 'w') as f:
        f.write("""from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP as TorchTHP
from easy_tpp.model.torch_model.torch_nhp import NHP as TorchNHP
from easy_tpp.model.torch_model.torch_rothp import RoTHP as TorchRoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP as TorchHoTHP
""")

    sys.path.insert(0, os.path.abspath('ufc-easytpp'))
    os.system('pip install omegaconf -q')
else:
    ROOT = os.path.abspath('..')
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)

CKPT_DIR = pathlib.Path('checkpoints_time_scale')
CKPT_DIR.mkdir(exist_ok=True)

print('Setup OK')

In [ ]:
# ── Imports do easy_tpp ───────────────────────────────────────────────────────
import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention

import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

CORES = {
    'NHP':   '#55A868',
    'THP':   '#8172B2',
    'RoTHP': '#4C72B0',
    'HoTHP': '#C44E52',
}

MODELOS = [
    ('NHP',   NHP),
    ('THP',   THP),
    ('RoTHP', RoTHP),
    ('HoTHP', HoTHP),
]

print(f'Device: {device}')
print('Imports OK')

In [ ]:
# ── Configuracoes ─────────────────────────────────────────────────────────────
TRAIN_LEN      = 50
EXTRAP_FACTORS = [1, 2, 5, 10]
N_SEEDS        = 5
EPOCHS         = 300
PATIENCE       = 30
NUM_TYPES      = 2
PAD_ID         = NUM_TYPES

REGIMES = {
    'Decaimento lento  (beta=0.025)': dict(
        mu    = np.array([0.4, 0.4]),
        alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
        beta  = 0.025,
    ),
    'Decaimento rapido (beta=0.50)': dict(
        mu    = np.array([0.4, 0.4]),
        alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
        beta  = 0.50,
    ),
}

LR_POR_MODELO = {
    'NHP':   1e-3,
    'THP':   1e-3,
    'RoTHP': 1e-3,
    'HoTHP': 5e-4,
}

def make_config():
    return ModelConfig(**{
        'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
        'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
        'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
        'gpu': 0 if torch.cuda.is_available() else -1,
        'model_id': 'Val',
        'model_specs': {},
        'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                     'patience_counter': 5, 'num_samples_boundary': 5,
                     'dtime_max': 5.0, 'num_step_gen': 1},
        'loss_integral_num_sample_per_step': 20,
        'use_mc_samples': False,
    })

print(f'TRAIN_LEN={TRAIN_LEN}, fatores={EXTRAP_FACTORS}, seeds={N_SEEDS}')

In [ ]:
# ── Funcoes auxiliares ────────────────────────────────────────────────────────

def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(200):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch, pad_id=PAD_ID):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad = torch.zeros(B, L)
    d_pad = torch.zeros(B, L)
    k_pad = torch.full((B, L), pad_id, dtype=torch.long)
    npm   = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl] = 1.0
        m = causal.clone()
        m[:, sl:] = True; m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(data, batch_size=min(bs, len(data)), shuffle=shuffle,
                      collate_fn=collate, generator=g)


def eval_nll(model, dl):
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            l, n = model.loglike_loss(batch)
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def eval_prediction(model, dl):
    model.eval()
    all_dt_err_sq, all_type_correct, all_count = [], [], []
    with torch.no_grad():
        for batch in dl:
            batch_dev = [t.to(device) for t in batch]
            time_seqs, time_delta_seqs, type_seqs, batch_non_pad_mask, attention_mask = batch_dev
            try:
                dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch_dev)
            except Exception:
                return float('nan'), float('nan')
            dt_label = time_delta_seqs[:, 1:]
            type_label = type_seqs[:, 1:]
            mask = batch_non_pad_mask[:, 1:]
            min_len = min(dtimes_pred.size(1), dt_label.size(1))
            dtimes_pred = dtimes_pred[:, :min_len]
            types_pred = types_pred[:, :min_len]
            dt_label = dt_label[:, :min_len]
            type_label = type_label[:, :min_len]
            mask = mask[:, :min_len]
            dt_err_sq = ((dtimes_pred - dt_label) ** 2) * mask
            type_correct = ((types_pred == type_label).float()) * mask
            n = mask.sum().item()
            all_dt_err_sq.append(dt_err_sq.sum().item())
            all_type_correct.append(type_correct.sum().item())
            all_count.append(n)
    total_n = sum(all_count)
    if total_n == 0:
        return float('nan'), float('nan')
    return math.sqrt(sum(all_dt_err_sq) / total_n), sum(all_type_correct) / total_n


def extract_hope_params(model):
    """Extrai os parametros aprendidos do HoTHP. Retorna dict ou None se nao for HoTHP."""
    if not hasattr(model, 'hope_emb'):
        return None
    emb = model.hope_emb
    return {
        'log_time_scale': emb.log_time_scale.item(),
        'time_scale':     emb.time_scale.item(),
        'theta_prime_raw': emb.theta_prime_raw.item(),
        'theta_prime':    emb.theta_prime.item(),
        'thetas_max':     emb.thetas.max().item(),
        'thetas_min':     emb.thetas.min().item(),
    }


def train_model(cls, train_dl, val_dl, lr, seed):
    set_seed(seed)
    config = make_config()
    m = cls(config).to(device)
    opt   = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=10, min_lr=1e-5)
    best_val, best_state, no_imp = float('inf'), None, 0

    # Historico de parametros HoPE por epoca
    hope_history = []

    for ep in range(EPOCHS):
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            l, n = m.loglike_loss(batch)
            nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                nll.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
        v = eval_nll(m, val_dl)
        sched.step(v)

        # Registrar parametros HoPE a cada epoca
        hp = extract_hope_params(m)
        if hp is not None:
            hp['epoch'] = ep
            hp['val_nll'] = v
            hope_history.append(hp)

        if v < best_val - 1e-4:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break

    m.load_state_dict(best_state)
    return m, best_val, hope_history


# ── Checkpoints ───────────────────────────────────────────────────────────────

def ckpt_path(model_name, regime_idx, seed_idx):
    return CKPT_DIR / f'{model_name}_regime{regime_idx}_seed{seed_idx}.pt'


def save_checkpoint(model, model_name, regime_idx, seed_idx, val_nll, hope_history):
    path = ckpt_path(model_name, regime_idx, seed_idx)
    torch.save({
        'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
        'model_name': model_name,
        'val_nll': val_nll,
        'hope_history': hope_history,
    }, path)
    return path


def load_checkpoint(cls, model_name, regime_idx, seed_idx):
    path = ckpt_path(model_name, regime_idx, seed_idx)
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location=device, weights_only=False)
    config = make_config()
    m = cls(config).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    hope_history = ckpt.get('hope_history', [])
    return m, ckpt['val_nll'], hope_history


USE_CHECKPOINTS = True

print('Funcoes prontas.')

In [ ]:
# ── Experimento principal ─────────────────────────────────────────────────────

METRIC_NAMES = ['nll', 'rmse', 'acc']
MODEL_NAMES = [name for name, _ in MODELOS]

# all_results[regime][factor][model_name][metric] = list of values per seed
all_results = {}

# hope_params[regime][seed_idx] = {'final': dict, 'history': list}
all_hope_params = {}

# hope_histories[regime][seed_idx] = list of dicts (one per epoch)
all_hope_histories = {}

for regime_idx, (regime_name, proc) in enumerate(REGIMES.items()):
    print(f'\n{"="*70}')
    print(f'Regime: {regime_name}')
    print(f'{"="*70}')

    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']
    horizon_train = max(50.0, TRAIN_LEN / mu.sum() * 3)

    results = {
        f: {mn: {met: [] for met in METRIC_NAMES} for mn in MODEL_NAMES}
        for f in EXTRAP_FACTORS
    }
    hope_params_regime = {}
    hope_histories_regime = {}

    for seed_idx in range(N_SEEDS):
        seed = 42 + seed_idx * 100
        rng  = np.random.default_rng(seed)

        raw_train = [simulate_hawkes(rng, mu, alpha, beta, horizon_train, 5, TRAIN_LEN)
                     for _ in range(400)]
        raw_val   = [simulate_hawkes(rng, mu, alpha, beta, horizon_train, 5, TRAIN_LEN)
                     for _ in range(100)]

        train_dl = make_loader(to_tensors(raw_train), 64, shuffle=True, seed=seed)
        val_dl   = make_loader(to_tensors(raw_val),   64)

        trained = {}
        for model_name, model_cls in MODELOS:
            loaded = None
            if USE_CHECKPOINTS:
                loaded = load_checkpoint(model_cls, model_name, regime_idx, seed_idx)

            if loaded is not None:
                model, val_nll, hope_history = loaded
                print(f'  [{model_name}] seed {seed_idx}: carregado do checkpoint (val={val_nll:.4f})')
            else:
                model, val_nll, hope_history = train_model(
                    model_cls, train_dl, val_dl,
                    lr=LR_POR_MODELO[model_name],
                    seed=seed + hash(model_name) % 1000)
                save_checkpoint(model, model_name, regime_idx, seed_idx, val_nll, hope_history)
                print(f'  [{model_name}] seed {seed_idx}: treinado  (val={val_nll:.4f})', end='')

                # Mostrar parametros HoPE se for HoTHP
                hp = extract_hope_params(model)
                if hp is not None:
                    print(f'  time_scale={hp["time_scale"]:.6f}  theta_prime={hp["theta_prime"]:.4f}', end='')
                print()

            trained[model_name] = model

            # Guardar parametros HoPE finais
            if model_name == 'HoTHP':
                hp_final = extract_hope_params(model)
                hope_params_regime[seed_idx] = hp_final
                hope_histories_regime[seed_idx] = hope_history

        # Avalia em cada fator de extrapolacao
        for f in EXTRAP_FACTORS:
            target_len = TRAIN_LEN * f
            horizon_test = max(horizon_train * f, horizon_train + 10)
            raw_test = [simulate_hawkes(rng, mu, alpha, beta, horizon_test,
                                        TRAIN_LEN + 1, target_len)
                        for _ in range(200)]
            test_dl = make_loader(to_tensors(raw_test), 32)

            row = []
            for model_name, _ in MODELOS:
                model = trained[model_name]
                nll_val = eval_nll(model, test_dl)
                rmse_val, acc_val = eval_prediction(model, test_dl)
                results[f][model_name]['nll'].append(nll_val)
                results[f][model_name]['rmse'].append(rmse_val)
                results[f][model_name]['acc'].append(acc_val)
                row.append(f'{model_name}: NLL={nll_val:.4f}')
            print(f'    a={f:>2}x  ' + '  |  '.join(row))

    all_results[regime_name] = results
    all_hope_params[regime_name] = hope_params_regime
    all_hope_histories[regime_name] = hope_histories_regime

print('\nExperimento concluido.')

In [ ]:
# ── Tabela: parametros HoPE aprendidos por regime ────────────────────────────

print('=' * 90)
print('PARAMETROS HoPE APRENDIDOS — HoTHP')
print('=' * 90)

for regime_name, params in all_hope_params.items():
    print(f'\n--- {regime_name} ---')
    print(f'{"seed":>6} {"time_scale":>12} {"log_ts":>10} {"theta_prime":>12} {"tp_raw":>10} {"thetas_max":>12}')
    print('-' * 68)
    ts_vals, tp_vals = [], []
    for seed_idx in sorted(params.keys()):
        hp = params[seed_idx]
        ts_vals.append(hp['time_scale'])
        tp_vals.append(hp['theta_prime'])
        print(f'{seed_idx:>6} {hp["time_scale"]:>12.6f} {hp["log_time_scale"]:>10.4f} '
              f'{hp["theta_prime"]:>12.4f} {hp["theta_prime_raw"]:>10.4f} {hp["thetas_max"]:>12.4f}')
    ts_arr = np.array(ts_vals)
    tp_arr = np.array(tp_vals)
    print(f'{"media":>6} {ts_arr.mean():>12.6f} {"":>10} {tp_arr.mean():>12.4f}')
    print(f'{"std":>6} {ts_arr.std():>12.6f} {"":>10} {tp_arr.std():>12.4f}')

In [ ]:
# ── Tabela resumo NLL ─────────────────────────────────────────────────────────

print('RESUMO — NLL medio +/- desvio (n={} seeds)'.format(N_SEEDS))
print()

for regime_name, results in all_results.items():
    print(f'{regime_name}')
    header = f'  {"a":>4}'
    for mn in MODEL_NAMES:
        header += f'  {mn:>14}'
    print(header)
    print(f'  {"---":>4}' + f'  {"---":>14}' * len(MODEL_NAMES))
    for f in EXTRAP_FACTORS:
        row = f'  {f:>3}x'
        for mn in MODEL_NAMES:
            vals = np.array(results[f][mn]['nll'])
            row += f'  {vals.mean():.4f}+/-{vals.std():.4f}'
        print(row)
    print()

In [ ]:
# ── Grafico 1: Evolucao do time_scale e theta_prime durante treinamento ───────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (regime_name, histories) in enumerate(all_hope_histories.items()):
    # time_scale
    ax_ts = axes[0, col]
    for seed_idx, hist in sorted(histories.items()):
        if not hist:
            continue
        epochs = [h['epoch'] for h in hist]
        ts_vals = [h['time_scale'] for h in hist]
        ax_ts.plot(epochs, ts_vals, alpha=0.6, lw=1.5, label=f'seed {seed_idx}')
    ax_ts.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='time_scale = 1.0')
    ax_ts.set_ylabel('time_scale = exp(log_time_scale)', fontsize=11)
    ax_ts.set_title(regime_name, fontsize=11, fontweight='bold')
    ax_ts.legend(fontsize=8)
    ax_ts.grid(True, alpha=0.3)
    ax_ts.spines['top'].set_visible(False)
    ax_ts.spines['right'].set_visible(False)

    # theta_prime
    ax_tp = axes[1, col]
    for seed_idx, hist in sorted(histories.items()):
        if not hist:
            continue
        epochs = [h['epoch'] for h in hist]
        tp_vals = [h['theta_prime'] for h in hist]
        ax_tp.plot(epochs, tp_vals, alpha=0.6, lw=1.5, label=f'seed {seed_idx}')
    # Mostrar thetas_max como referencia
    if hist:
        thetas_max = hist[0]['thetas_max']
        ax_tp.axhline(y=thetas_max, color='red', linestyle=':', alpha=0.5,
                      label=f'max(theta_j) = {thetas_max:.4f}')
    ax_tp.set_xlabel('Epoca', fontsize=11)
    ax_tp.set_ylabel("theta' (coef. decaimento global)", fontsize=11)
    ax_tp.legend(fontsize=8)
    ax_tp.grid(True, alpha=0.3)
    ax_tp.spines['top'].set_visible(False)
    ax_tp.spines['right'].set_visible(False)

fig.suptitle('Evolucao dos parametros HoPE durante treinamento do HoTHP\n'
             f'(n={N_SEEDS} seeds, {EPOCHS} epocas max, patience={PATIENCE})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('time_scale_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grafico 2: Evolucao conjunta time_scale + val_nll ────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, (regime_name, histories) in enumerate(all_hope_histories.items()):
    ax = axes[col]
    for seed_idx, hist in sorted(histories.items()):
        if not hist:
            continue
        epochs = [h['epoch'] for h in hist]
        ts_vals = [h['time_scale'] for h in hist]
        nll_vals = [h['val_nll'] for h in hist]

        color = plt.cm.tab10(seed_idx)
        ax.plot(epochs, nll_vals, '-', color=color, alpha=0.7, lw=1.5, label=f'NLL seed {seed_idx}')

        # Anotar o time_scale final
        ax.annotate(f'ts={ts_vals[-1]:.3f}',
                    xy=(epochs[-1], nll_vals[-1]),
                    fontsize=7, color=color, ha='left')

    ax.set_xlabel('Epoca', fontsize=11)
    ax.set_ylabel('Validation NLL', fontsize=11)
    ax.set_title(regime_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Convergencia do HoTHP — NLL de validacao com time_scale final anotado',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('time_scale_vs_nll.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grafico 3: Parametros finais — comparacao entre regimes ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

regime_names = list(all_hope_params.keys())
regime_short = ['Lento\n(beta=0.025)', 'Rapido\n(beta=0.50)']

# time_scale por regime (boxplot)
ax = axes[0]
data_ts = []
for rn in regime_names:
    vals = [all_hope_params[rn][s]['time_scale'] for s in sorted(all_hope_params[rn].keys())]
    data_ts.append(vals)
bp = ax.boxplot(data_ts, labels=regime_short, patch_artist=True,
                boxprops=dict(facecolor='#C44E52', alpha=0.4),
                medianprops=dict(color='#C44E52', lw=2))
for i, vals in enumerate(data_ts):
    ax.scatter([i+1]*len(vals), vals, color='#C44E52', zorder=3, s=40, alpha=0.7)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='init = 1.0')
ax.set_ylabel('time_scale (aprendido)', fontsize=11)
ax.set_title('time_scale final por regime', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# theta_prime por regime (boxplot)
ax = axes[1]
data_tp = []
for rn in regime_names:
    vals = [all_hope_params[rn][s]['theta_prime'] for s in sorted(all_hope_params[rn].keys())]
    data_tp.append(vals)
bp = ax.boxplot(data_tp, labels=regime_short, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.4),
                medianprops=dict(color='#4C72B0', lw=2))
for i, vals in enumerate(data_tp):
    ax.scatter([i+1]*len(vals), vals, color='#4C72B0', zorder=3, s=40, alpha=0.7)
# Mostrar thetas_max
first_regime = regime_names[0]
first_seed = sorted(all_hope_params[first_regime].keys())[0]
thetas_max = all_hope_params[first_regime][first_seed]['thetas_max']
ax.axhline(y=thetas_max, color='red', linestyle=':', alpha=0.5,
           label=f'max(theta_j) = {thetas_max:.4f}')
ax.set_ylabel("theta' (aprendido)", fontsize=11)
ax.set_title("theta' final por regime", fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle(f'Parametros HoPE aprendidos — HoTHP (n={N_SEEDS} seeds)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hope_params_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grafico 4: NLL por fator de extrapolacao (mesma viz do notebook original) ─

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(EXTRAP_FACTORS))
xlabels = [f'{f}x' for f in EXTRAP_FACTORS]

for ax, (regime_name, results) in zip(axes, all_results.items()):
    for mn in MODEL_NAMES:
        means = np.array([np.mean(results[f][mn]['nll']) for f in EXTRAP_FACTORS])
        stds  = np.array([np.std(results[f][mn]['nll'])  for f in EXTRAP_FACTORS])
        ax.plot(x, means, 'o-', color=CORES[mn], lw=2, ms=7, label=mn)
        ax.fill_between(x, means - stds, means + stds, color=CORES[mn], alpha=0.12)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_xlabel('Fator de extrapolacao (a)', fontsize=12)
    ax.set_ylabel('NLL (nats)', fontsize=12)
    ax.set_title(regime_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(f'NLL — Train Short, Test Long (TRAIN_LEN={TRAIN_LEN}, n={N_SEEDS} seeds)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('time_scale_nll_extrapolation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grafico 5: Consolidado 3 metricas x 2 regimes ───────────────────────────

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

metric_info = [
    ('nll',  'NLL (nats)',  'o-', 'lower is better'),
    ('rmse', 'RMSE (dt)',   's-', 'lower is better'),
    ('acc',  'Accuracy',    '^-', 'higher is better'),
]

for row, (met_key, ylabel, marker, note) in enumerate(metric_info):
    for col, (regime_name, results) in enumerate(all_results.items()):
        ax = axes[row, col]
        for mn in MODEL_NAMES:
            raw = [results[f][mn][met_key] for f in EXTRAP_FACTORS]
            means = np.array([np.nanmean(v) for v in raw])
            stds  = np.array([np.nanstd(v) for v in raw])
            if np.all(np.isnan(means)):
                continue
            ax.plot(x, means, marker, color=CORES[mn], lw=2, ms=7, label=mn)
            ax.fill_between(x, means - stds, means + stds, color=CORES[mn], alpha=0.12)

        ax.set_xticks(x)
        ax.set_xticklabels(xlabels)
        ax.set_xlabel('Fator de extrapolacao (a)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        title = regime_name if row == 0 else ''
        if title:
            ax.set_title(title, fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if col == 0:
            ax.annotate(f'({note})', xy=(0.02, 0.95), xycoords='axes fraction',
                        fontsize=8, color='gray', va='top')

fig.suptitle(f'Validacao time_scale — Train Short, Test Long\n'
             f'TRAIN_LEN={TRAIN_LEN}, n={N_SEEDS} seeds, {len(MODEL_NAMES)} modelos',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('time_scale_consolidado.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Salvar resultados em JSON ─────────────────────────────────────────────────

def results_to_serializable(all_results):
    out = {}
    for regime, factors in all_results.items():
        out[regime] = {}
        for f, models in factors.items():
            out[regime][str(f)] = {}
            for mn, metrics in models.items():
                out[regime][str(f)][mn] = {
                    met: [float(v) for v in vals] for met, vals in metrics.items()
                }
    return out

def hope_params_to_serializable(all_hope_params):
    out = {}
    for regime, seeds in all_hope_params.items():
        out[regime] = {}
        for seed_idx, hp in seeds.items():
            out[regime][str(seed_idx)] = {k: float(v) for k, v in hp.items()}
    return out

save_data = {
    'results': results_to_serializable(all_results),
    'hope_params': hope_params_to_serializable(all_hope_params),
    'config': {
        'TRAIN_LEN': TRAIN_LEN,
        'EXTRAP_FACTORS': EXTRAP_FACTORS,
        'N_SEEDS': N_SEEDS,
        'EPOCHS': EPOCHS,
        'PATIENCE': PATIENCE,
    },
}

with open(CKPT_DIR / 'results_time_scale.json', 'w') as f:
    json.dump(save_data, f, indent=2)
print(f'Resultados salvos em {CKPT_DIR / "results_time_scale.json"}')

## Analise e conclusoes

**Preencher apos execucao:**

1. **`time_scale` aprendido — regime lento vs rapido:**
   - O `time_scale` converge para valores diferentes entre regimes?
   - Se sim: o modelo adapta a escala temporal interna ao regime do processo
   - Se ~1.0 em ambos: os dados ja chegam normalizados (to_tensors divide por mean gap) e o parametro nao precisa compensar

2. **`theta_prime` aprendido:**
   - Converge para valores consistentes entre seeds?
   - Se estavel: o decaimento global do kernel hiperbolico e robusto
   - Se varia muito: o parametro e mal-condicionado ou redundante com time_scale

3. **Evolucao durante treinamento:**
   - O time_scale estabiliza cedo ou oscila ate o final?
   - Correlacao entre estabilizacao do time_scale e convergencia da NLL?

4. **Comparacao com modelos sem time_scale (NHP, THP, RoTHP):**
   - O HoTHP (com time_scale aprendido e sem clamp) e competitivo?
   - O parametro extra justifica a complexidade adicional?